In [24]:
import pandas as pd 
import numpy as na 
import unicodedata 
import re

In [126]:
bill_details= pd.read_csv('MA_Bill_Details.csv')
legislature_members= pd.read_csv('MA_legislature_members.csv')
roll_calls= pd.read_csv('MA_Senate_rollcalls.csv')
bill_rollcalls= pd.read_csv('MA_Bill_rollcall_mapping.csv')


In [79]:
print(bill_details.head())
print(legislature_members.head())
print(roll_calls.head())
print(bill_rollcalls.head())

   GeneralCourtNumber BillNumber DocketNumber PrimarySponsorId  \
0                 183        NaN       SD2779              NaN   
1                 183        NaN       SD2780              NaN   
2                 183        NaN       SD2781              NaN   
3                 183        NaN       SD2782              NaN   
4                 183        NaN       SD2786              NaN   

  PrimarySponsorName                                              Title  
0                NaN  A report of the Executive Office of Transporta...  
1                NaN  Communication from the Hampden County Sheriff’...  
2                NaN  Communication from the Hampshire County Sherif...  
3                NaN  A report of the Special Transportation Finance...  
4                NaN  A report of the Massachusetts School Building ...  
   GeneralCourtNumber MemberCode                  FullName  \
0                 194       DAF0        Dylan A. Fernandes   
1                 194       KAD0   

## Data Cleaning

### Bills

In [4]:
bill_details.head()

,GeneralCourtNumber,BillNumber,DocketNumber,PrimarySponsorId,PrimarySponsorName,Title
0,183,NaN,SD2779,NaN,NaN,A report of the Executive Office of Transporta...
1,183,NaN,SD2780,NaN,NaN,Communication from the Hampden County Sheriff’...
2,183,NaN,SD2781,NaN,NaN,Communication from the Hampshire County Sherif...
3,183,NaN,SD2782,NaN,NaN,A report of the Special Transportation Finance...
4,183,NaN,SD2786,NaN,NaN,A report of the Massachusetts School Building ...


In [5]:
bill_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80271 entries, 0 to 80270
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  80271 non-null  int64 
 1   BillNumber          69214 non-null  object
 2   DocketNumber        72449 non-null  object
 3   PrimarySponsorId    70200 non-null  object
 4   PrimarySponsorName  79216 non-null  object
 5   Title               80050 non-null  object
dtypes: int64(1), object(5)
memory usage: 3.7+ MB


In [ ]:
bill_details['GeneralCourtNumber'].value_counts()

In [ ]:

bill_details_clean = bill_details.copy()

# Clean text fields
for col in ['BillNumber', 'DocketNumber', 'PrimarySponsorId', 'PrimarySponsorName', 'Title']:
    bill_details_clean[col] = (
        bill_details_clean[col]
        .astype(str)
        .str.strip()
        .replace({'nan': pd.NA})
    )

bill_details_clean = bill_details_clean.drop_duplicates()

# Remove rows missing all major identifiers
bill_details_clean = bill_details_clean.dropna(subset=['BillNumber'], how='all')

bill_details_clean['GeneralCourtNumber'] = bill_details_clean['GeneralCourtNumber'].astype(int)

print(bill_details_clean.info())


<class 'pandas.core.frame.DataFrame'>
Index: 69214 entries, 9 to 80267
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  69214 non-null  int64 
 1   BillNumber          69214 non-null  object
 2   DocketNumber        61392 non-null  object
 3   PrimarySponsorId    67566 non-null  object
 4   PrimarySponsorName  68618 non-null  object
 5   Title               69181 non-null  object
dtypes: int64(1), object(5)
memory usage: 3.7+ MB
None


In [22]:
bill_details_clean.head()

,GeneralCourtNumber,BillNumber,DocketNumber,PrimarySponsorId,PrimarySponsorName,Title
9,184,H4176,HD94176,<NA>,<NA>,relative to HIV and Hepatitis C prevention
11,185,S2742,SD92742,<NA>,<NA>,An Act relative to landscape sprinkler systems
12,185,S2783,SD92783,S48,Senate Committee on Post Audit and Oversight,A report of the committee on Post Audit and Ov...
13,185,S2870,SD92870,S48,Senate Committee on Post Audit and Oversight,The Cost of Inaction: Climate Change in the Co...
14,186,S99,SD1,BET0,Bruce E. Tarr,An Act to provide for loans for waterfront dev...


### Legislature members

In [25]:
legislature_members.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2773 entries, 0 to 2772
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   GeneralCourtNumber  2773 non-null   int64  
 1   MemberCode          2773 non-null   object 
 2   FullName            2768 non-null   object 
 3   District            2143 non-null   object 
 4   Branch              2769 non-null   object 
 5   Party               2662 non-null   object 
 6   Active              0 non-null      float64
 7   LeadershipPosition  369 non-null    object 
dtypes: float64(1), int64(1), object(6)
memory usage: 173.4+ KB


In [23]:
legislature_members.head()

,GeneralCourtNumber,MemberCode,FullName,District,Branch,Party,Active,LeadershipPosition
0,194,DAF0,Dylan A. Fernandes,Plymouth and Barnstable,Senate,Democrat,NaN,NaN
1,194,KAD0,Kelly A. Dooner,Third Bristol and Plymouth,Senate,Republican,NaN,Assistant Minority Leader
2,194,WJD0,"William J. Driscoll, Jr.","Norfolk, Plymouth and Bristol",Senate,Democrat,NaN,NaN
3,194,MLB1,Michelle L. Badger,1st Plymouth,House,Democrat,NaN,NaN
4,194,MSC1,Michael S. Chaisson,1st Bristol,House,Republican,NaN,NaN


In [47]:
def clean_text(s):
    if pd.isna(s):
        return s
    # accent
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    # punctuation
    s = re.sub(r'[^\w\s]', '', s)
    # spaces and lowercase
    s=re.sub(r'\s+', ' ', s).strip().title()
    return s

def text_normalize(df):
    cols_to_check = ['FullName', 'District', 'Branch', 'Party']

# normalizing name - removing extra characters, replace name with longest when a member has multiple values in name
    df['FullName_clean'] = df['FullName'].apply(clean_text)
    df['FullName_length'] = df['FullName_clean'].fillna('').apply(len)
    longest_name = (
        df.loc[df.groupby('MemberCode')['FullName_length'].idxmax(), ['MemberCode', 'FullName_clean']]
        .rename(columns={'FullName_clean': 'LongestFullName'})
    )
    # Merge back — replace all names for that MemberCode
    df = df.merge(longest_name, on='MemberCode', how='left')
    df['FullName'] = df['LongestFullName']
    df.drop(columns=['FullName_clean', 'FullName_length', 'LongestFullName'], inplace=True)

# normalizing district
    df['District_clean'] = df['District'].apply(clean_text)
    df['District_length'] = df['District_clean'].fillna('').apply(len)
    longest_district = (
        df.loc[df.groupby('MemberCode')['District_length'].idxmax(), ['MemberCode', 'District_clean']]
        .rename(columns={'District_clean': 'LongestDistrict'})
    )
    # Merge back — replace all districts for that MemberCode
    df = df.merge(longest_district, on='MemberCode', how='left')
    df['District'] = df['LongestDistrict']
    df.drop(columns=['District_clean', 'District_length', 'LongestDistrict'], inplace=True)

    return df



legislature_members_clean = legislature_members.copy()

text_cols = ['MemberCode', 'FullName', 'District', 'Branch', 'Party', 'LeadershipPosition']
for col in text_cols:
    legislature_members_clean[col] = (
        legislature_members_clean[col]
        .astype(str)
        .str.strip()
        .replace({'nan': pd.NA, 'NaN': pd.NA})
    )
legislature_members_clean= text_normalize( legislature_members_clean)

legislature_members_clean= legislature_members_clean.drop('Active', axis=1)

legislature_members_clean = legislature_members_clean.drop_duplicates(subset=['GeneralCourtNumber', 'MemberCode'], keep='first')

legislature_members_clean['GeneralCourtNumber'] = legislature_members_clean['GeneralCourtNumber'].astype(int)

print(legislature_members_clean.info())



<class 'pandas.core.frame.DataFrame'>
Index: 2696 entries, 0 to 2772
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  2696 non-null   int64 
 1   MemberCode          2696 non-null   object
 2   FullName            2692 non-null   object
 3   District            2336 non-null   object
 4   Branch              2692 non-null   object
 5   Party               2585 non-null   object
 6   LeadershipPosition  347 non-null    object
dtypes: int64(1), object(6)
memory usage: 168.5+ KB
None


In [30]:
print(legislature_members_clean['Branch'].value_counts())
print(legislature_members_clean['Party'].value_counts())
print(legislature_members_clean['LeadershipPosition'].value_counts())

Branch
House     2135
Senate     546
Joint       11
Name: count, dtype: int64
Party
Democrat       2152
Republican      417
Independent       9
Unenrolled        7
Name: count, dtype: int64
LeadershipPosition
Assistant Majority Leader             41
Second Assistant Majority Leader      37
Third Assistant Minority Leader       31
Minority Leader                       26
Majority Leader                       24
Fourth Division Chair                 19
Second Assistant Minority Leader      19
Assistant Minority Leader             18
Third Division Chair                  16
President of the Senate               15
House Ways and Means Chair            14
Senate President Emerita              10
Speaker of the House                   9
Senate Majority Whip                   9
Speaker Pro Tempore                    9
First Assistant Minority Leader        8
President Pro Tempore                  8
First Division Chair                   8
Second Division Chair                  7
Assistant Ma

In [48]:
legislature_members_clean.head()

,GeneralCourtNumber,MemberCode,FullName,District,Branch,Party,LeadershipPosition
0,194,DAF0,Dylan A Fernandes,Plymouth And Barnstable,Senate,Democrat,<NA>
1,194,KAD0,Kelly A Dooner,Third Bristol And Plymouth,Senate,Republican,Assistant Minority Leader
2,194,WJD0,William J Driscoll Jr,Norfolk Plymouth And Bristol,Senate,Democrat,<NA>
3,194,MLB1,Michelle L Badger,1St Plymouth,House,Democrat,<NA>
4,194,MSC1,Michael S Chaisson,1St Bristol,House,Republican,<NA>


### Roll calls

In [51]:
roll_calls.head()


,GeneralCourtNumber,Branch,RollCallNumber,MemberCode,MemberURL,Vote
0,188,Senate,2,MJB0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
1,188,Senate,2,SMB0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
2,188,Senate,2,WNB0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
3,188,Senate,2,GDC0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
4,188,Senate,2,HLC0,http://malegislature.gov/api/GeneralCourts/188...,Yeas


In [52]:
roll_calls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99031 entries, 0 to 99030
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  99031 non-null  int64 
 1   Branch              99031 non-null  object
 2   RollCallNumber      99031 non-null  int64 
 3   MemberCode          98851 non-null  object
 4   MemberURL           98851 non-null  object
 5   Vote                99031 non-null  object
dtypes: int64(2), object(4)
memory usage: 4.5+ MB


In [ ]:
roll_calls_clean = roll_calls.copy()

string_cols = ['Branch', 'MemberCode', 'MemberURL', 'Vote']
for col in string_cols:
    roll_calls_clean[col] = (
        roll_calls_clean[col]
        .astype(str)
        .str.strip()
        .replace({'nan': pd.NA, 'NaN': pd.NA})
    )

# Normalizing Vote values 
roll_calls_clean['Vote'] = (
    roll_calls_clean['Vote']
    .replace({
        'Yeas': 'Y',
        'Nays':'N',
        'Present': 'P',
        'Absent': 'A'
    })
)

roll_calls_clean = roll_calls_clean.drop_duplicates(subset=['GeneralCourtNumber', 'Branch', 'RollCallNumber', 'MemberCode'])

roll_calls_clean['GeneralCourtNumber'] = roll_calls_clean['GeneralCourtNumber'].astype(int)
roll_calls_clean['RollCallNumber'] = roll_calls_clean['RollCallNumber'].astype(int)

print(roll_calls_clean.info())


<class 'pandas.core.frame.DataFrame'>
Index: 98991 entries, 0 to 99030
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  98991 non-null  int64 
 1   Branch              98991 non-null  object
 2   RollCallNumber      98991 non-null  int64 
 3   MemberCode          98811 non-null  object
 4   MemberURL           98811 non-null  object
 5   Vote                98991 non-null  object
dtypes: int64(2), object(4)
memory usage: 5.3+ MB
None


In [57]:
roll_calls_clean.head()

,GeneralCourtNumber,Branch,RollCallNumber,MemberCode,MemberURL,Vote
0,188,Senate,2,MJB0,http://malegislature.gov/api/GeneralCourts/188...,Y
1,188,Senate,2,SMB0,http://malegislature.gov/api/GeneralCourts/188...,Y
2,188,Senate,2,WNB0,http://malegislature.gov/api/GeneralCourts/188...,Y
3,188,Senate,2,GDC0,http://malegislature.gov/api/GeneralCourts/188...,Y
4,188,Senate,2,HLC0,http://malegislature.gov/api/GeneralCourts/188...,Y


In [77]:
bill_details_clean.head()

,GeneralCourtNumber,BillNumber,DocketNumber,PrimarySponsorId,PrimarySponsorName,Title
9,184,H4176,HD94176,<NA>,<NA>,relative to HIV and Hepatitis C prevention
11,185,S2742,SD92742,<NA>,<NA>,An Act relative to landscape sprinkler systems
12,185,S2783,SD92783,S48,Senate Committee on Post Audit and Oversight,A report of the committee on Post Audit and Ov...
13,185,S2870,SD92870,S48,Senate Committee on Post Audit and Oversight,The Cost of Inaction: Climate Change in the Co...
14,186,S99,SD1,BET0,Bruce E. Tarr,An Act to provide for loans for waterfront dev...


In [75]:
max_rollcalls= roll_calls_clean.groupby('GeneralCourtNumber')['RollCallNumber'].max()
court_rollcalls= roll_calls_clean.groupby('GeneralCourtNumber')['RollCallNumber'].apply(set).to_dict()

for i in max_rollcalls.items():
    court_no, max_rollcall= i[0], i[1]
    missing=[]
    for j in range(1,max_rollcall):
        if j not in court_rollcalls[court_no]:
            missing.append(j)
    if len(missing):
        print(f'General court: {court_no}, Missing roll calls: {missing}')



General court: 188, Missing roll calls: [1, 136, 147, 195, 385, 432, 467, 474]
General court: 189, Missing roll calls: [1, 95, 238, 299, 300, 301, 302, 303]
General court: 190, Missing roll calls: [1, 119, 314, 387, 452, 453, 454, 481]
General court: 191, Missing roll calls: [1, 93, 122, 321]
General court: 192, Missing roll calls: [1, 47]
General court: 193, Missing roll calls: [195]
General court: 194, Missing roll calls: [1]


## Bill-Rollcall mapping

In [127]:
bill_rollcalls.head()


,GeneralCourtNumber,BillNumber,RollCallNumber
0,186,H1696,293
1,186,H4156,348
2,186,H4316,295
3,186,H4368,291
4,186,H4470,190


In [128]:
bill_rollcalls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2808 entries, 0 to 2807
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  2808 non-null   int64 
 1   BillNumber          2808 non-null   object
 2   RollCallNumber      2808 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 65.9+ KB


In [129]:
bill_rollcalls_clean = bill_rollcalls.copy()

string_cols = ['BillNumber']
for col in string_cols:
    bill_rollcalls_clean[col] = (
        bill_rollcalls_clean[col]
        .astype(str)
        .str.strip()
        .replace({'nan': pd.NA, 'NaN': pd.NA})
    )

bill_rollcalls_clean = bill_rollcalls_clean.drop_duplicates()
print(bill_rollcalls_clean.info())


<class 'pandas.core.frame.DataFrame'>
Index: 2804 entries, 0 to 2807
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  2804 non-null   int64 
 1   BillNumber          2804 non-null   object
 2   RollCallNumber      2804 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 87.6+ KB
None


# Final Denormalized dataset

In [130]:
for df in [bill_details_clean, legislature_members_clean, roll_calls_clean]:
    df['GeneralCourtNumber'] = df['GeneralCourtNumber'].astype(int)

# Join rollcalls with members
rollcalls_members = roll_calls_clean.merge(
    legislature_members_clean,
    on=['GeneralCourtNumber', 'MemberCode'],
    how='left',
    validate='m:1'  
)

# Join rollcalls with bills to get BillId
rollcalls_bills = rollcalls_members.merge(
    bill_rollcalls_clean,
    on=['GeneralCourtNumber', 'RollCallNumber'],
    how='left',
    validate='m:1'  
)

# Join with bill details
final_rollcall_df = rollcalls_bills.merge(
    bill_details_clean,
    on=['GeneralCourtNumber', 'BillNumber'],
    how='left',
    # validate='m:1'  
)

final_rollcall_df.drop(columns='Branch_y', inplace=True)
final_rollcall_df.rename(columns={'Branch_x':'Branch'}, inplace=True)
final_rollcall_df.reset_index(drop=True, inplace=True)
final_rollcall_df.info()


MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

In [121]:
# Missing rollcall and bill mapping
missing_bill = final_rollcall_check[final_rollcall_df['BillNumber'].isna()]
missing_summary = (
    missing_bill[['GeneralCourtNumber', 'RollCallNumber']]
    .drop_duplicates()
    .sort_values(['GeneralCourtNumber', 'RollCallNumber'])
)
missing_summary

/var/folders/tn/sjn_hw5j1nj7llqb8csdkb0r0000gn/T/ipykernel_19092/571760479.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  missing_bill = final_rollcall_check[final_rollcall_df['BillNumber'].isna()]


,GeneralCourtNumber,RollCallNumber
56630,191,14
58824,191,66
69020,191,302
69260,191,308
69300,191,309


In [87]:

# Group by GeneralCourtNumber and BillNumber, count rows
bill_counts = (
    bill_details_clean.groupby(['GeneralCourtNumber', 'BillNumber'])
      .size()
      .reset_index(name='RowCount')
)

print(bill_counts[bill_counts['RowCount']>1])


       GeneralCourtNumber BillNumber  RowCount
10449                 187      H3399         2
10672                 187        H36         2
13525                 187      S2071         2


In [107]:
final_rollcall_check

,GeneralCourtNumber,Branch,RollCallNumber,MemberCode,MemberURL,Vote,FullName,District,Party,LeadershipPosition,BillNumber,DocketNumber,PrimarySponsorId,PrimarySponsorName,Title
86381,193,Senate,1,MFR0,http://malegislature.gov/api/GeneralCourts/193...,A,Michael F Rush,Norfolk And Suffolk,Democrat,Senate Majority Whip,NaN,NaN,NaN,NaN,NaN
86382,193,Senate,2,BET0,http://malegislature.gov/api/GeneralCourts/193...,Y,Bruce E Tarr,First Essex And Middlesex,Republican,Minority Leader,S17,<NA>,TS29,Temporary Senate Committee on Rules,Proposed Senate Rules for the 2023-2024 Legisl...
86383,193,Senate,2,JFK0,http://malegislature.gov/api/GeneralCourts/193...,Y,John F Keenan,Norfolk And Plymouth,Democrat,<NA>,S17,<NA>,TS29,Temporary Senate Committee on Rules,Proposed Senate Rules for the 2023-2024 Legisl...
86384,193,Senate,2,PMO,http://malegislature.gov/api/GeneralCourts/193...,Y,Patrick M Oconnor,First Plymouth And Norfolk,Republican,Assistant Minority Leader,S17,<NA>,TS29,Temporary Senate Committee on Rules,Proposed Senate Rules for the 2023-2024 Legisl...
86385,193,Senate,2,RCF0,http://malegislature.gov/api/GeneralCourts/193...,Y,Ryan C Fattman,Worcester And Hampden,Republican,Assistant Minority Leader,S17,<NA>,TS29,Temporary Senate Committee on Rules,Proposed Senate Rules for the 2023-2024 Legisl...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98986,194,Senate,74,WNB0,http://malegislature.gov/api/GeneralCourts/194...,Y,William N Brownsberger,Second Suffolk And Middlesex,Democrat,President Pro Tempore,S2653,<NA>,<NA>,<NA>,An Act updating the Move Over Law
98987,194,Senate,74,LME0,http://malegislature.gov/api/GeneralCourts/194...,Y,Lydia Edwards,First Suffolk And Middlesex,Democrat,<NA>,S2653,<NA>,<NA>,<NA>,An Act updating the Move Over Law
98988,194,Senate,74,A_G0,http://malegislature.gov/api/GeneralCourts/194...,Y,Adam Gomez,Hampden,Democrat,<NA>,S2653,<NA>,<NA>,<NA>,An Act updating the Move Over Law
98989,194,Senate,74,JJC0,http://malegislature.gov/api/GeneralCourts/194...,Y,John J Cronin,Worcester And Middlesex,Democrat,<NA>,S2653,<NA>,<NA>,<NA>,An Act updating the Move Over Law
